# FLUX.1-dev LoRA Training with Ostris AI-Toolkit

This notebook sets up the environment and trains a FLUX.1-dev LoRA on a 24GB VRAM GPU (like an RTX 4090 on RunPod).

## 1. Environment Setup & Auto-Restart

Cloning the repository, installing dependencies, and restarting the kernel to ensure the new packages are loaded.

In [ ]:
!git clone https://github.com/ostris/ai-toolkit.git
%cd ai-toolkit
!git submodule update --init --recursive
!pip install -r requirements.txt
!pip install huggingface_hub

# CRITICAL: Restart kernel programmatically
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

### Hugging Face Authentication

Please replace `'YOUR_HF_TOKEN_HERE'` with your actual Hugging Face token (with read permissions for FLUX.1-dev).

In [ ]:
from huggingface_hub import login

# Replace with your token
login(token='YOUR_HF_TOKEN_HERE')

## 2. Dataset Directory Setup & Verification

Creating dataset and output directories. The script below will verify that every image has a corresponding `.txt` caption file.

In [ ]:
import os
import glob

dataset_dir = "/workspace/dataset"
output_dir = "/workspace/output"

os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print(f"Dataset directory created at: {dataset_dir}")
print(f"Output directory created at: {output_dir}")

def verify_dataset(directory):
    image_exts = {'.jpg', '.jpeg', '.png'}
    images = []
    
    for ext in image_exts:
        images.extend(glob.glob(os.path.join(directory, f'*{ext}')))
        images.extend(glob.glob(os.path.join(directory, f'*{ext.upper()}')))
        
    mismatches = 0
    for img_path in images:
        base_name = os.path.splitext(img_path)[0]
        txt_path = base_name + '.txt'
        if not os.path.exists(txt_path):
            print(f"WARNING: Missing caption file for {img_path}")
            mismatches += 1
            
    if mismatches == 0 and len(images) > 0:
        print(f"Success: Found {len(images)} images, all with matching .txt caption files.")
    elif len(images) == 0:
        print("Dataset directory is empty. Please add your images and captions.")

verify_dataset(dataset_dir)

## 3. Dynamic YAML Configuration Generation

Define your dataset variables below. We enforce hyperparameters safe for 24GB VRAM (FP8 base model offloading, 8-bit Adam, batch size 1).

In [ ]:
import yaml
import os

# ==========================================
# DEFINE YOUR VARIABLES HERE
# ==========================================
DATASET_NAME = 'vga_founder'
TRIGGER_WORD = 'vga_founder'
SAMPLE_PROMPTS = [
    f"A portrait of {TRIGGER_WORD} in a cyber city",
    f"{TRIGGER_WORD} as an astronaut on Mars",
    f"A close up shot of {TRIGGER_WORD} looking at the camera"
]
# ==========================================

config_dir = "/workspace/ai-toolkit/config"
os.makedirs(config_dir, exist_ok=True)

job_config = {
    "job": "extension",
    "config": {
        "name": DATASET_NAME,
        "process": [
            {
                "type": "sd_trainer",
                "training_folder": "/workspace/output",
                "device": "cuda:0",
                "network": {
                    "type": "lora",
                    "linear": 16,
                    "linear_alpha": 16
                },
                "save": {
                    "dtype": "bfloat16",
                    "save_every": 500,
                    "max_step_saves_to_keep": 1
                },
                "datasets": [
                    {
                        "folder_path": "/workspace/dataset",
                        "caption_ext": "txt",
                        "caption_dropout_rate": 0.05,
                        "shuffle_tokens": False,
                        "cache_latents_to_disk": True,
                        "resolution": [512, 768, 1024]
                    }
                ],
                "train": {
                    "batch_size": 1,
                    "steps": 1500,
                    "gradient_accumulation_steps": 1,
                    "train_unet": True,
                    "train_text_encoder": False,
                    "gradient_checkpointing": True,
                    "noise_scheduler": "flowmatch",
                    "optimizer": "adamw8bit",
                    "lr": 4e-4,
                    "ema_config": {
                        "use_ema": True,
                        "ema_decay": 0.99
                    },
                    "dtype": "bfloat16"
                },
                "model": {
                    "name_or_path": "black-forest-labs/FLUX.1-dev",
                    "is_flux": True,
                    "quantize": True
                },
                "sample": {
                    "sampler": "flowmatch",
                    "sample_every": 500,
                    "width": 1024,
                    "height": 1024,
                    "prompts": SAMPLE_PROMPTS,
                    "neg": "",
                    "seed": 42,
                    "walk_seed": True,
                    "guidance_scale": 4,
                    "sample_steps": 20
                }
            }
        ],
        "meta": {
            "name": DATASET_NAME,
            "version": "1.0"
        }
    }
}

yaml_path = os.path.join(config_dir, "job.yaml")
with open(yaml_path, 'w') as f:
    yaml.dump(job_config, f, sort_keys=False)

print(f"Generated job configuration at {yaml_path}")

## 4. Training Execution

Run the training. Note the `%cd /workspace/ai-toolkit` ensuring we are in the correct directory after the kernel restart.

In [ ]:
%cd /workspace/ai-toolkit
!python run.py config/job.yaml

## 5. Export & Cleanup

Find the most recently created `.safetensors` file in the output directory and zip it up for easy downloading.

In [ ]:
import os
import glob
import zipfile

output_dir = "/workspace/output"
zip_path = "/workspace/flux_lora_final.zip"

safetensors = glob.glob(os.path.join(output_dir, "**", "*.safetensors"), recursive=True)

if not safetensors:
    print("No .safetensors files found in the output directory.")
else:
    # Find the most recently modified file
    latest_safetensor = max(safetensors, key=os.path.getmtime)
    print(f"Most recent safetensor found: {latest_safetensor}")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(latest_safetensor, arcname=os.path.basename(latest_safetensor))
        
    print(f"Successfully zipped the LoRA to: {zip_path}")
